[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/symmetry/symmetry.ipynb)

# Material Symmetry

This example computes the constitutive responses that a symmetric crystal allows by averaging candidate laws over its point group ([Neumann's principle](https://dictionary.iucr.org/Neumann%27s_principle)). Because any physical response of the crystal must look unchanged after every symmetry of its lattice, the group mean projects an arbitrary response onto the invariant part permitted by the symmetry. Under the proper rotations of a cube, heat conduction becomes completely isotropic while elastic stiffness retains cubic anisotropy; under mirror reflections and spatial inversion, optical rotary power reverses sense and cancels unless improper rotations pair orthogonal directions of opposite hand.

Each material law is modelled as a typed multilinear extensor: conductivity maps driving planes to heat-flow directions via `axes * (Field & axes)`, elasticity evaluates four strain-reading planes via `plane & bond`, and optical gyration maps two light directions to an intrinsic rotary-power scalar via `(planes & Direction) * (planes & Direction) * powers`. Every symmetry versor acts uniformly on input and output by sandwiching, `versor >> conductivity(versor << Field)` and `versor >> gyration(versor << Direction, versor << Direction)`, so odd versors automatically flip the intrinsic rotary power while leaving conductivity untouched.

In crystal-physics tensor notation, heat conduction, elasticity, and optical gyration are written as Cartesian component arrays $K_{ij}$, $C_{ijkl}$, and $g_{ij}$ transformed under orthogonal point-group matrices $R_{ia}$ as $K'_{ij} = R_{ia} R_{jb} K_{ab}$ and $C'_{ijkl} = R_{ia} R_{jb} R_{kc} R_{ld} C_{abcd}$, while axial-tensor bookkeeping inserts an explicit determinant factor $g'_{ij} = \det(R)\, R_{ia} R_{jb}\, g_{ab}$ to track improper rotations.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
import numpy as np

from numga import Algebra, Extensor, NumpyContext, concatenate, stack
from examples.util.animation import save_figure
from examples.mechanics.symmetry import render, scenarios

np.set_printoptions(precision=4, suppress=True)

ga = Algebra("x+y+z+")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                     # Scalar+
# Driving fields (such as a temperature slope) and planes that read bond stretch or light directions are
# extrinsic planes; heat flows, bonds, and light paths are intrinsic directions.
Field = ga.gatype.vector()                                                      # Vector+
Plane = ga.gatype.vector()                                                      # Vector+
Direction = ga.gatype.antivector()                                              # Antivector-
# A polarization turn per unit distance carries an intrinsic sense of rotation.
RotaryPower = ga.gatype.antipseudoscalar()                                      # Antipseudoscalar-
Conductivity = ga.gatype((Direction, Field))                                    # Direction- <- Field+
Elasticity = ga.gatype((Scalar, Plane, Plane, Plane, Plane))                    # Scalar+ <- (Plane+, Plane+, Plane+, Plane+)
Gyration = ga.gatype((RotaryPower, Direction, Direction))                       # RotaryPower- <- (Direction-, Direction-)
# Turning planes, proper rotations, and improper rotations (reflections and spatial inversion).
TurningPlane = ga.gatype.bivector()                                             # Bivector+
Rotor = ga.gatype.rotor()                                                       # Rotor+
ImproperRotation = ga.gatype.odd()                                              # Odd+

## Heat conduction under rotation groups

A positive conductivity with three distinct principal gains along a tilted frame is averaged over three proper rotation groups: orthorhombic half turns in two orthogonal planes, tetragonal quarter turns in the horizontal plane, and the 24 rotations of a cube. The six independent components of a symmetric conductivity drop to three, two, and one as the symmetry group grows.

In [ ]:
pose = (mv.xy * scenarios.pose_xy + mv.yz * scenarios.pose_yz).exp()            # [] Rotor+
principal_axes = pose >> stack((mv.x.dual(), mv.y.dual(), mv.z.dual()))         # [3] Direction-
gains = scenarios.gains
driving = (mv.x + mv.y + mv.z).normalized()                                     # [] Field+

In [ ]:
def turns(plane: TurningPlane, n_turns: int) -> Rotor:
    """The cyclic group of `n_turns` equal rotations (`[n_turns] Rotor+`) in `plane` starts at the identity."""
    # A rotor turns by twice its bivector angle, so `pi * k / n_turns` turns by `2 * pi * k / n_turns`.
    return (plane * (np.pi * np.arange(n_turns) / n_turns)).exp()               # [n_turns] Rotor+


def cube_rotations() -> Rotor:
    """The 24 proper rotations of a cube (`[24] Rotor+`) bring each of the six faces to the top before four quarter turns."""
    tilt = np.pi / 4 * np.array([0, 1, 2, 3, 0, 0])
    roll = np.pi / 4 * np.array([0, 0, 0, 0, 1, -1])
    faces = (mv.yz * tilt + mv.zx * roll).exp()                                 # [6] Rotor+
    return (faces[:, None] * turns(mv.xy, 4)[None, :]).reshape(-1)              # [24] Rotor+


cube = cube_rotations()                                                         # [24] Rotor+
orthorhombic_turns = (turns(mv.yz, 2)[:, None] * turns(mv.xy, 2)[None, :]).reshape(-1)  # [4] Rotor+
conduction_groups = [orthorhombic_turns, turns(mv.xy, 4), cube]

In [ ]:
def conductivities(axes: Direction, gains: np.ndarray, groups: list[Rotor]) -> Conductivity:
    """A positive conductivity along `axes` and its Reynolds averages over `groups` form `[1 + len(groups)]` maps (`Direction- <- Field+`).

    Conductivity maps a driving field to a heat flow that need not align with the field in an anisotropic crystal.
    Averaging `rotor >> conductivity(rotor << Field)` over a finite rotation group projects onto the closest
    group-invariant positive conductivity in the Frobenius norm.
    """
    # Each principal axis pairs with the driving field and carries heat flow along itself.
    conductivity = (axes * (Field & axes) * gains).sum(axis=0)                  # [] Direction- <- Field+
    # Pulling the field into each rotated frame and turning the resulting flow back leaves the group mean invariant.
    return Extensor.stack([conductivity] + [(group >> conductivity(group << Field)).mean(axis=0) for group in groups])


responses = conductivities(principal_axes, gains, conduction_groups)            # [4] Direction- <- Field+
fluxes = responses(driving)                                                     # [4] Direction-
save_figure(render.conduction_figure(responses, driving, fluxes), "symmetry_conductivity");

From left to right, each translucent blue surface shows the heat-flow ellipsoid of unit driving fields for the tilted candidate and its orthorhombic, tetragonal, and cubic averages, while the dark arrow marks one driving field and the orange arrow marks the heat flow it drives, aligning with the field once cube symmetry makes the ellipsoid a sphere.

In [ ]:
# Pairing the unit driving field with the normalized heat flow gives the cosine of the deflection angle between them.
cos_deflection = (driving & fluxes) / fluxes.norm()                             # [4] Scalar+
print("angle between the driving field and the heat flow (deg):", np.degrees(np.arccos(cos_deflection.to_array())))
print("cubic conductivity:\n", responses[-1].coefficient_table())

## Conduction and elasticity of a cubic lattice

The 24 cube rotations carry one axial seed bond and one face-diagonal seed bond onto every bond of a simple-cubic lattice. Averaging bond conduction `bonds * (Field & bonds)` over the group gives the same scalar gain in every direction, whereas averaging the fourth-order spring energy `(Plane & bonds) ** 4` keeps a [cubic stiffness](https://www.ctcms.nist.gov/oof/oof1/Manual/node152.html) that peaks across the cube faces and dips along the body diagonals.

In [ ]:
seed_bonds = mv.vector(scenarios.seed_bond_coords).dual().normalized()          # [2] Direction-
centre, axial, diagonal, corners = scenarios.lattice_shells()
sphere_directions, unit_fields = scenarios.sphere_directions, scenarios.unit_fields

In [ ]:
def lattice_responses(seed_bonds: Direction, cube: Rotor) -> tuple[Conductivity, Elasticity]:
    """Averaging `seed_bonds` over `cube` yields the simple-cubic lattice conductivity and four-plane elasticity.

    The 24 cube rotations carry axial and face-diagonal seed bonds onto every bond of their families with equal
    weight, so another cube rotation permutes the terms in the group average. At unit lattice spacing with axial
    spring stiffness 1/3 and face-diagonal stiffness 1/12, both families carry unit total weight and satisfy the
    central-force Cauchy relation `c12 == c44`.

    A uniaxial strain of magnitude `s` across a unit plane stretches a unit bond by `s * (plane & bond) ** 2`, so
    the quadratic spring energy defines a symmetric four-plane form `elasticity(plane, plane, plane, plane)` holding
    transverse strain fixed: https://www.ctcms.nist.gov/oof/oof1/Manual/node152.html
    """
    bonds = cube[:, None] >> seed_bonds[None, :]                                # [..., 2] Direction-
    # Each bond paired with a plane gives the directional stretch factor whose fourth power enters spring energy.
    stretch_readings = Plane & bonds                                            # [..., 2] Scalar+ <- Plane+

    # Averaging over the group and summing the two families makes conduction isotropic while keeping cubic elasticity.
    conductivity = (bonds * (Field & bonds)).mean(axis=0).sum(axis=0)           # [] Direction- <- Field+
    elasticity = (stretch_readings * stretch_readings * stretch_readings * stretch_readings).mean(axis=0).sum(axis=0)  # [] Scalar+ <- (Plane+, Plane+, Plane+, Plane+)
    return conductivity, elasticity


lattice_conductivity, elasticity = lattice_responses(seed_bonds, cube)
# Evaluating the four-plane elasticity on each unit plane gives the directional uniaxial stiffness plotted radially.
probes = unit_fields                                                            # [33, 65] Plane+
stiffness_surface = sphere_directions * elasticity(probes, probes, probes, probes)  # [33, 65] Direction-
save_figure(render.crystal_figure(centre, axial, diagonal, corners, lattice_conductivity, stiffness_surface), "symmetry_crystal_lattice");

The left panel shows the central lattice site in dark grey with its orange axial bonds, blue face-diagonal bonds, and pale corner neighbours; the middle panel shows the resulting spherical conduction surface; and the right panel plots uniaxial stiffness as a radius, bulging to 1/2 along the cube faces and indenting to 1/3 along the body diagonals.

In [ ]:
cube_face, octahedral_face = mv.x, (mv.x + mv.y + mv.z).normalized()            # [] Plane+, Plane+
face_stiffness = elasticity(cube_face, cube_face, cube_face, cube_face)         # [] Scalar+
diagonal_stiffness = elasticity(octahedral_face, octahedral_face, octahedral_face, octahedral_face)  # [] Scalar+
print("uniaxial stiffness across a cube face and a body diagonal:", face_stiffness.to_array(), diagonal_stiffness.to_array())

## Optical activity under proper and improper groups

Optical rotary power turns a light beam's polarization plane with a definite sense, so it takes values in the intrinsic scalars and reverses sign under odd versors. Averaging a tilted gyration map over proper rotations and improper point groups separates chiral crystals, achiral non-centrosymmetric crystals that turn light with opposite senses along mirror-related axes, and centrosymmetric crystals in which spatial inversion cancels rotary power in every direction.

In [ ]:
principal_planes = pose >> stack((mv.x, mv.y, mv.z))                            # [3] Plane+
powers = mv(RotaryPower, scenarios.rotary_values[:, None])                      # [3] RotaryPower-

In [ ]:
def rotary_powers(planes: Plane, powers: RotaryPower, groups: list[tuple[Rotor, ImproperRotation]]) -> Gyration:
    """A candidate gyration map on `planes` and its point-group averages over `groups` form `[1 + len(groups)]` maps (`RotaryPower- <- (Direction-, Direction-)`).

    Light travelling along a unit direction `d` has its polarization plane turned per unit distance by `gyration(d, d)`.
    Because rotary power is an intrinsic scalar, improper rotations reverse its sense when carrying the output back,
    cancelling optical activity completely whenever the point group contains spatial inversion.
    """
    gyration = ((planes & Direction) * (planes & Direction) * powers).sum(axis=0)  # [] RotaryPower- <- (Direction-, Direction-)

    def group_average(rotations: Rotor, improper_rotations: ImproperRotation) -> Gyration:
        # Improper versors flip the intrinsic sense of `gyration` when sandwiching the output back to the crystal frame.
        return concatenate([
            versor >> gyration(versor << Direction, versor << Direction)
            for versor in (rotations, improper_rotations)
        ]).mean(axis=0)

    return Extensor.stack([gyration] + [group_average(rotations, improper_rotations) for rotations, improper_rotations in groups])


# Spatial inversion is the unit pseudoscalar acting as an odd improper rotation.
inversion = mv.xyz                                                              # [] ImproperRotation+
quarter_turns = turns(mv.xy, 4)                                                 # [4] Rotor+
point_groups = [
    (quarter_turns, inversion * quarter_turns[:0]),
    (quarter_turns, inversion * quarter_turns),
    (quarter_turns[::2], inversion * quarter_turns[1::2]),
    (cube, inversion * cube[:0]),
    (cube, inversion * cube),
]
gyrations = rotary_powers(principal_planes, powers, point_groups)               # [6] RotaryPower- <- (Direction-, Direction-)
rotary_surfaces = gyrations[:, None, None](sphere_directions[None, :, :], sphere_directions[None, :, :])  # [6, 33, 65] RotaryPower-
save_figure(render.optical_activity_figure(gyrations), "symmetry_optical_activity");

Red and blue shade opposite senses of polarization rotation across the sphere of light directions for the tilted candidate, pure fourfold rotations, fourfold rotations with inversion, fourfold rotoinversion, pure cube rotations, and cube rotations with inversion; white spheres mark the two centrosymmetric groups where inversion eliminates optical activity, while the fourth sphere alternates red and blue lobes across the horizontal axes.

In [ ]:
print("maximum rotary power per point group:", rotary_surfaces[1:].norm().to_array().max(axis=(1, 2)))

In [ ]:
# checks
# Cube rotations make conduction isotropic at the mean gain; simple-cubic bond conduction is 2/3 while stiffness is cubic;
# spatial inversion cancels rotary power, fourfold rotoinversion alternates senses, and cube rotations leave the mean rotary power.
assert ((responses[-1](unit_fields) - sphere_directions * gains.mean()).norm() < 1e-7).all()
assert (((mv.x & responses[2](mv.x)) - (mv.y & responses[2](mv.y))).abs() < 1e-6).all()
assert (((unit_fields & lattice_conductivity(unit_fields)) - 2 / 3).abs() < 1e-7).all()
assert ((face_stiffness - 1 / 2).abs() < 1e-7).all()
assert ((diagonal_stiffness - 1 / 3).abs() < 1e-7).all()
for centrosymmetric in (rotary_surfaces[2], rotary_surfaces[5]):
    assert (centrosymmetric.norm() < 1e-9).all()
assert ((rotary_surfaces[4] - powers.mean(axis=0)).norm() < 1e-9).all()
x_dir, y_dir, z_dir = mv.x.dual(), mv.y.dual(), mv.z.dual()
assert ((gyrations[3](x_dir, x_dir) + gyrations[3](y_dir, y_dir)).norm() < 1e-9).all()
assert (gyrations[3](z_dir, z_dir).norm() < 1e-9).all()